# 12 — OE Superpixel Glint Correction

Copy of `11_lsq_superpixel_inversion.ipynb` with pure LSQ replaced by full OE
(Optimal Estimation with prior).  Uses `superpixel_engine.invert_image_superpixel`
with the default `invert_fn=None` → `dask_engine.invert_image`.

Pipeline:
1. **Segment** the Rrs image into spectrally coherent superpixels (SLIC)
2. **Invert** the mean spectrum of each superpixel with OE (prior + data term)
3. **Back-interpolate** to full pixel resolution via PCA + kNN IDW
   - `x_hat` and `A_diag`: IDW average of k nearest superpixels
   - `sigma`: IDW-propagated posterior + interpolation spread

**Chi-2 calibration note (OE + superpixels):**  
`invert_superpixels` passes the pixel-level noise unchanged.  Mean spectra are
smoother (variance σ²/N), so `chi2_raw` ≈ chi2_per_pixel / N.  
The calibrated chi2 stored in the output is `chi2_calibrated = chi2_raw × N`;  
its ideal value is **≈ 1** (same target as per-pixel OE).

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import scipy.ndimage as ndi
import lmfit
import time
import xarray as xr
import rioxarray
import jax
import jax.numpy as jnp
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser
from skimage.segmentation import mark_boundaries

from bio_optics.coupled_models import albert_mobley_3C_jax
from bio_optics.inversion import oe_engine
from bio_optics.image_processing import dask_engine, superpixel_engine
from bio_optics.surface import air_water
from bio_optics.surface.reflectance import Rrs_surf
from bio_optics.atmosphere import sky_radiance

jax.config.update('jax_enable_x64', True)

## Parameters and forward model

Same 3C model as NB11, with OE priors (`sigma_a_3C`) added.

In [ ]:
params_3C = lmfit.Parameters()
params_3C.add('C_0',   value=2,          min=0,     max=100, vary=True)
params_3C.add('C_1',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_2',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_3',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_4',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_5',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_Y',   value=0.2,        min=0,     max=4,   vary=True)
params_3C.add('C_X',   value=5,          min=0,     max=100, vary=True)
params_3C.add('C_Mie', value=1,          min=0,     max=100, vary=True)
params_3C.add('f_0',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_1',   value=1,          min=0,     max=1,   vary=False)
params_3C.add('f_2',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_3',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_4',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_5',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('B_0',   value=1/np.pi,                        vary=False)
params_3C.add('B_1',   value=1/np.pi,                        vary=False)
params_3C.add('B_2',   value=1/np.pi,                        vary=False)
params_3C.add('B_3',   value=1/np.pi,                        vary=False)
params_3C.add('B_4',   value=1/np.pi,                        vary=False)
params_3C.add('B_5',   value=1/np.pi,                        vary=False)
params_3C.add('bb_phy_spec',         value=0.0010,            vary=False)
params_3C.add('bb_Mie_spec',         value=0.0042,            vary=False)
params_3C.add('bb_X_spec',           value=0.0086,            vary=False)
params_3C.add('a_NAP_spec_lambda_0', value=0.041,             vary=False)
params_3C.add('S',                   value=0.014,             vary=False)
params_3C.add('K',                   value=0,                 vary=False)
params_3C.add('S_NAP',               value=0.011,             vary=False)
params_3C.add('n',                   value=-1,                vary=False)
params_3C.add('lambda_0',            value=440,               vary=False)
params_3C.add('lambda_S',            value=500,               vary=False)
params_3C.add('theta_sun',  value=np.radians(30),             vary=False)
params_3C.add('theta_view', value=np.radians(1e-10),          vary=False)
params_3C.add('n1',    value=1,                               vary=False)
params_3C.add('n2',    value=1.33,                            vary=False)
params_3C.add('kappa_0', value=1.0546,                        vary=False)
params_3C.add('zB',    value=100,  min=0.1, max=1000,         vary=False)
params_3C.add('T_W',   value=18,   min=0,   max=40,           vary=False)
params_3C.add('T_W_0', value=20,                              vary=False)
params_3C.add('g_dd',  value=0.02, min=0,   max=10,           vary=True)
params_3C.add('g_dsr', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('g_dsa', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('d_r',   value=0.01, min=0,   max=0.1,          vary=True)
params_3C.add('fd_d',  value=1,                               vary=False)
params_3C.add('fd_s',  value=1,                               vary=False)
params_3C.add('offset', value=0,  min=0,    max=0.01,         vary=False)

sigma_a_3C = {
    'C_0':   2.0,   # log-space: prior=2,    ±2σ range (0.3, 15)
    'C_Y':   1.5,   # log-space: prior=0.2,  ±2σ range (0.04, 0.9)
    'C_X':   2.0,   # log-space: prior=5,    ±2σ range (0.7, 37)
    'C_Mie': 2.3,   # log-space: prior=1,    ±2σ range (0.1, 10)
    'g_dd':  3.1,   # log-space: prior=0.02, ±2σ range (0.0004, 0.44)
    'g_dsr': 1.7,   # log-space: prior=1/π,  ±2σ range (0.06, 1.8)
    'g_dsa': 1.7,   # log-space: prior=1/π,  ±2σ range (0.06, 1.8)
    'd_r':   1.15,  # log-space: prior=0.01, ±2σ range (0.003, 0.032)
}
log_params_3C = ['C_0', 'C_Y', 'C_X', 'C_Mie', 'g_dd', 'g_dsr', 'g_dsa', 'd_r']

NOISE     = 0.0031   # sr⁻¹ — update so median chi2_calibrated ≈ 1
TILE_SIZE = 4096
N_ITER    = 15       # dask_engine Gauss-Newton iterations

# SLIC + back-interpolation parameters
N_SEGMENTS   = 5000
COMPACTNESS  = 0.1
SLIC_SIGMA   = 2.0
K            = 4
N_COMPONENTS = 6

## Glint forward model (2D) — identical to NB10/NB11

In [ ]:
def forward_glint_2D(fit_param_ds, parameters, wavelengths, pre_3C):
    n2     = float(parameters['n2'].value)
    rho_L  = air_water.fresnel(parameters['theta_view'].value,
                               n1=parameters['n1'].value, n2=n2)
    Ed_d  = np.array(pre_3C['Ed_d'],  dtype=float)
    Ed_sr = np.array(pre_3C['Ed_sr'], dtype=float)
    Ed_sa = np.array(pre_3C['Ed_sa'], dtype=float)
    Ls_Ed = np.array(pre_3C['Ls_Ed'], dtype=float)
    Ed    = Ed_d + Ed_sr + Ed_sa

    def _get(name):
        if name in fit_param_ds:
            return fit_param_ds[name].values
        return fit_param_ds['x_hat'].sel(param=name).values

    g_dd  = _get('g_dd')[...,  np.newaxis]
    g_dsr = _get('g_dsr')[..., np.newaxis]
    g_dsa = _get('g_dsa')[..., np.newaxis]
    d_r   = _get('d_r')[...,   np.newaxis]

    L_s = sky_radiance.L_s(
        float(parameters['fd_d'].value), g_dd, Ed_d,
        float(parameters['fd_s'].value), g_dsr, Ed_sr,
        g_dsa, Ed_sa,
    )
    R_rs_surface  = Rrs_surf(L_s, Ed, rho_L, d_r)
    R_rs_surface += air_water.fresnel(float(parameters['theta_view'].value), n2=n2) * Ls_Ed
    return R_rs_surface.transpose(2, 0, 1)

## Data store

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

SCENE_IDS = [
    'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z',
]

## Run — SLIC superpixel OE glint correction

In [ ]:
for scene_id in SCENE_IDS:
    print(f'Processing {scene_id}')

    img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
    scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
    wavelengths = img.wavelength.values[:80]

    refl  = img['reflectance'].isel(band=slice(0, 80))
    rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
    cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
    rrs   = rrs.where(~cloud)

    MIN_WATER_PIXELS = 100
    _wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
    _wf = _wc['water_fraction'].values
    _labeled, _ = ndi.label(_wf >= 0.5)
    _sizes = np.bincount(_labeled.ravel())
    _sizes[0] = 0
    ocean_mask = xr.DataArray(
        np.isin(_labeled, np.where(_sizes >= MIN_WATER_PIXELS)[0]),
        coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
    )
    rrs = rrs.where(ocean_mask)

    pre_3C   = albert_mobley_3C_jax.precompute(
        wavelengths, fresh=False,
        theta_sun=float(params_3C['theta_sun'].value),
        P=1013.25, AM=1, RH=60, H_oz=0.38, WV=2.5, alpha=1.317, beta=0.2606,
    )
    f_vec_3C = albert_mobley_3C_jax.make_forward_vec(list(params_3C.keys()), pre_3C)
    setup_3C = oe_engine.build_inversion(
        params_3C, f_vec_3C, sigma_a_3C, log_params=log_params_3C
    )

    Rrs_arr = rrs.transpose('y', 'x', 'band').values
    n_rows, n_cols, _ = Rrs_arr.shape
    n_water = int(np.isfinite(Rrs_arr).all(axis=-1).sum())
    print(f'  Scene: {n_rows}×{n_cols}  ({n_water} water pixels)')

    t0 = time.perf_counter()
    results_sp = superpixel_engine.invert_image_superpixel(
        Rrs_arr, setup_3C, NOISE,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SLIC_SIGMA,
        k=K,
        n_components=N_COMPONENTS,
        invert_fn=dask_engine.invert_image,
        store_sp_results=True,
        n_iter=N_ITER,
        tile_size=TILE_SIZE,
    )
    t_total = time.perf_counter() - t0

    n_segs = len(results_sp['sp_counts'])
    print(f'  Superpixel OE done in {t_total:.1f}s  ({n_segs} segments, {n_water/n_segs:.0f} px/seg)')

    fit_names = results_sp['fit_names']
    coords_2d = {'y': rrs.y, 'x': rrs.x}
    sp_dims   = ('y', 'x', 'param')
    sp_coords = {**coords_2d, 'param': fit_names}

    glint_ds = xr.Dataset({
        'x_hat':           xr.DataArray(results_sp['x_hat'],  dims=sp_dims, coords=sp_coords),
        'sigma':           xr.DataArray(results_sp['sigma'],  dims=sp_dims, coords=sp_coords),
        'A_diag':          xr.DataArray(results_sp['A_diag'], dims=sp_dims, coords=sp_coords),
        'chi2':            xr.DataArray(results_sp['chi2'],            dims=('y', 'x'), coords=coords_2d),
        'chi2_calibrated': xr.DataArray(results_sp['chi2_calibrated'], dims=('y', 'x'), coords=coords_2d),
        'H_info':          xr.DataArray(results_sp['H_info'],          dims=('y', 'x'), coords=coords_2d),
    }).rio.write_crs(scene_crs)
    store.write_data(glint_ds,
                     f'{OUTPUT_PREFIX}{scene_id}-glint_params_oe_sp.zarr', replace=True)

    glint_arr = forward_glint_2D(glint_ds, params_3C, wavelengths, pre_3C)
    glint_da  = xr.DataArray(glint_arr, coords=rrs.coords, dims=rrs.dims).rio.write_crs(scene_crs)
    Rrs_corr  = (rrs - glint_da).rio.write_crs(scene_crs)
    store.write_data(Rrs_corr.to_dataset(name='Rrs'),
                     f'{OUTPUT_PREFIX}{scene_id}-Rrs_oe_sp.zarr', replace=True)
    print(f'  Saved.')

## Segmentation visualisation

In [ ]:
_labels = results_sp['labels']
_n_segs = len(np.unique(_labels))

_rgb = Rrs_arr[..., [2, 1, 0]].copy()
_rgb = np.where(np.isfinite(_rgb), _rgb, np.nanmin(_rgb))
_p2, _p98 = np.nanpercentile(_rgb, [2, 98])
_rgb = np.clip((_rgb - _p2) / (_p98 - _p2 + 1e-12), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(_rgb, origin='upper')
axes[0].set_title('Rrs false colour')
axes[0].axis('off')

_img_bd = mark_boundaries(_rgb, _labels, color=(1, 1, 0), mode='thick')
axes[1].imshow(_img_bd, origin='upper')
axes[1].set_title(f'SLIC segments (n={_n_segs}, compactness={COMPACTNESS}, σ={SLIC_SIGMA})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

sp_c = results_sp['sp_counts']
print(f'Segments: {_n_segs}  |  pixels/seg: min={sp_c.min()}  median={np.median(sp_c):.0f}  max={sp_c.max()}')

## Diagnostics — noise calibration

`chi2_calibrated = chi2_raw × N`.  For OE the ideal target is **≈ 1**.

In [ ]:
import hvplot
import holoviews as hv
import hvplot.xarray

_scene = SCENE_IDS[0]
_gp    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_oe_sp.zarr')
_chi2_cal = _gp['chi2_calibrated'].values
_med      = float(np.nanmedian(_chi2_cal[np.isfinite(_chi2_cal)]))

noise_cal = NOISE * np.sqrt(_med)   # OE target = 1

print(f'NOISE = {NOISE:.5f} sr⁻¹')
print(f'Median chi2_calibrated = {_med:.3f}  →  calibrated NOISE = {noise_cal:.5f} sr⁻¹')
if abs(_med - 1.0) > 0.1:
    print(f'→ Update NOISE = {noise_cal:.5f} and re-run.')
else:
    print('✓ chi2_calibrated ≈ 1 — noise well-calibrated.')

## Maps

In [ ]:
_scene = SCENE_IDS[0]
_gp    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_oe_sp.zarr')
_valid = np.isfinite(_gp['x_hat']).any('param')
_x_hat    = _gp['x_hat'].where(_valid)
_sigma    = _gp['sigma'].where(_valid)
_A_diag   = _gp['A_diag'].where(_valid)
_chi2_cal = _gp['chi2_calibrated'].where(_valid)
_opts2 = dict(x='x', y='y', aspect='equal', width=280, height=280, robust=True)

p_chi2 = _chi2_cal.hvplot(cmap='RdYlGn_r', clim=(0, 3), title='chi2_calibrated (target ≈ 1)', **_opts2)
p_dfs  = _A_diag.mean('param').hvplot(cmap='viridis', clim=(0, 1), title='mean A_diag (DFS/n)', **_opts2)

p_gdd  = _x_hat.sel(param='g_dd').hvplot( cmap='viridis', title='g_dd',  **_opts2)
p_gdsr = _x_hat.sel(param='g_dsr').hvplot(cmap='viridis', title='g_dsr', **_opts2)
p_gdsa = _x_hat.sel(param='g_dsa').hvplot(cmap='viridis', title='g_dsa', **_opts2)
p_dr   = _x_hat.sel(param='d_r').hvplot(  cmap='viridis', title='d_r',   **_opts2)

p_sig_gdd = _sigma.sel(param='g_dd').hvplot(cmap='Purples', title='σ(g_dd)',  **_opts2)
p_sig_dr  = _sigma.sel(param='d_r').hvplot( cmap='Purples', title='σ(d_r)',   **_opts2)

hv.Layout([p_chi2, p_dfs, p_gdd, p_gdsr, p_gdsa, p_dr, p_sig_gdd, p_sig_dr]).cols(4)

## Comparison — OE superpixel vs LSQ superpixel (NB11)

In [ ]:
_scene  = SCENE_IDS[0]
_gp_oe  = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_oe_sp.zarr')
_gp_lsq = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq_sp.zarr')

_params_to_show = ['g_dd', 'g_dsr', 'g_dsa', 'd_r']
_opts3 = dict(x='x', y='y', aspect='equal', width=260, height=260, robust=True)

panels = []
for pname in _params_to_show:
    _oe  = _gp_oe['x_hat'].sel(param=pname)
    _lsq = _gp_lsq['x_hat'].sel(param=pname)
    _diff = _oe - _lsq
    panels += [
        _oe.hvplot(  title=f'{pname} OE sp',      **_opts3),
        _lsq.hvplot( title=f'{pname} LSQ sp',     **_opts3),
        _diff.hvplot(title=f'{pname} diff OE−LSQ', cmap='RdBu_r', **_opts3),
    ]

hv.Layout(panels).cols(3)

## Glint-corrected Rrs — OE vs LSQ superpixel

In [ ]:
_scene = SCENE_IDS[0]
_wl    = store.open_data(f'{INPUT_PREFIX}{_scene}.zarr').wavelength.values[:80]

_Rrs_oe  = store.open_data(f'{OUTPUT_PREFIX}{_scene}-Rrs_oe_sp.zarr')['Rrs']
_Rrs_lsq = store.open_data(f'{OUTPUT_PREFIX}{_scene}-Rrs_lsq_sp.zarr')['Rrs']
_Rrs_oe  = _Rrs_oe.where(np.isfinite(_Rrs_oe).any('band'))
_Rrs_lsq = _Rrs_lsq.where(np.isfinite(_Rrs_lsq).any('band'))

r_idx = int(np.argmin(np.abs(_wl - 670)))
g_idx = int(np.argmin(np.abs(_wl - 550)))
b_idx = int(np.argmin(np.abs(_wl - 460)))
_opts_rgb = dict(x='x', y='y', bands='band', framewise=True, aspect='equal',
                 width=460, height=460, robust=True)

p_oe  = _Rrs_oe.isel(band=[r_idx, g_idx, b_idx]).hvplot.rgb(title='Glint-corrected (OE superpixel)',  **_opts_rgb)
p_lsq = _Rrs_lsq.isel(band=[r_idx, g_idx, b_idx]).hvplot.rgb(title='Glint-corrected (LSQ superpixel)', **_opts_rgb)
(p_oe + p_lsq)

---
## Step 2 — Water quality retrieval (OE superpixel)

Uses the glint-corrected Rrs from Step 1 as input.
Full OE with tighter concentration priors.  Chi2 target ≈ 1.

In [ ]:
from bio_optics.water.reflectance import albert_mobley_jax

params_wq = lmfit.Parameters()
params_wq.add('C_0',   value=1.0,  min=1e-10, max=100, vary=True)
params_wq.add('C_1',   value=0.1,  min=1e-10, max=10,  vary=True)
params_wq.add('C_2',   value=0.1,  min=1e-10, max=10,  vary=True)
params_wq.add('C_3',   value=0.1,  min=1e-10, max=10,  vary=True)
params_wq.add('C_4',   value=0.1,  min=1e-10, max=10,  vary=True)
params_wq.add('C_5',   value=0.1,  min=1e-10, max=10,  vary=True)
params_wq.add('C_Y',   value=0.2,  min=1e-10, max=5,   vary=True)
params_wq.add('C_X',   value=0.1,  min=1e-10, max=5,   vary=True)
params_wq.add('C_Mie', value=1.0,  min=1e-10, max=100, vary=True)
params_wq.add('f_0',   value=0.5,  min=0, max=1, vary=True)
params_wq.add('f_1',   value=0.5,  min=0, max=1, vary=True)
params_wq.add('f_2',   value=0,    min=0, max=1, vary=False)
params_wq.add('f_3',   value=0,    min=0, max=1, vary=False)
params_wq.add('f_4',   value=0,    min=0, max=1, vary=False)
params_wq.add('f_5',   value=0,    min=0, max=1, vary=False)
params_wq.add('B_0',   value=1/np.pi, vary=False)
params_wq.add('B_1',   value=1/np.pi, vary=False)
params_wq.add('B_2',   value=1/np.pi, vary=False)
params_wq.add('B_3',   value=1/np.pi, vary=False)
params_wq.add('B_4',   value=1/np.pi, vary=False)
params_wq.add('B_5',   value=1/np.pi, vary=False)
params_wq.add('bb_phy_spec',         value=0.0010, vary=False)
params_wq.add('bb_Mie_spec',         value=0.0042, vary=False)
params_wq.add('bb_X_spec',           value=0.0086, vary=False)
params_wq.add('a_NAP_spec_lambda_0', value=0.041,  vary=False)
params_wq.add('S',                   value=0.014,  vary=False)
params_wq.add('K',                   value=0,      vary=False)
params_wq.add('S_NAP',               value=0.011,  vary=False)
params_wq.add('n',                   value=-1,     vary=False)
params_wq.add('lambda_0',            value=440,    vary=False)
params_wq.add('lambda_S',            value=500,    vary=False)
params_wq.add('theta_sun',  value=np.radians(30),    vary=False)
params_wq.add('theta_view', value=np.radians(1e-10), vary=False)
params_wq.add('n1',         value=1,                 vary=False)
params_wq.add('n2',         value=1.33,              vary=False)
params_wq.add('kappa_0',    value=1.0546,            vary=False)
params_wq.add('zB',    value=5.0, min=0.1, max=200, vary=True)
params_wq.add('T_W',   value=18,  min=0,   max=40,  vary=False)
params_wq.add('T_W_0', value=20,                    vary=False)

sigma_a_wq = {
    'C_0':  0.7, 'C_1': 0.7, 'C_2': 0.7, 'C_3': 0.7,
    'C_4':  0.7, 'C_5': 0.7, 'C_Y': 0.7, 'C_X': 0.7, 'C_Mie': 0.7,
    'f_0':  0.4, 'f_1': 0.4,
    'zB':   1.0,
}
log_params_wq = ['C_0', 'C_1', 'C_2', 'C_3', 'C_4', 'C_5', 'C_Y', 'C_X', 'C_Mie', 'zB']

NOISE_WQ = 0.0031   # sr⁻¹ — update so median chi2_calibrated ≈ 1

## Run — Step 2 OE superpixel water quality retrieval

In [ ]:
for scene_id in SCENE_IDS:
    print(f'Processing {scene_id}')

    _img        = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
    scene_crs   = _img.rio.crs or CRS.from_wkt(_img.spatial_ref.attrs['crs_wkt'])
    wavelengths = _img.wavelength.values[:80]

    _rrs_ds    = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-Rrs_oe_sp.zarr')
    _rrs_da    = _rrs_ds['Rrs']

    pre_wq   = albert_mobley_jax.precompute(wavelengths)
    f_vec_wq = albert_mobley_jax.make_forward_vec(list(params_wq.keys()), pre_wq)
    setup_wq = oe_engine.build_inversion(
        params_wq, f_vec_wq, sigma_a_wq, log_params=log_params_wq
    )

    Rrs_arr_wq = _rrs_da.transpose('y', 'x', 'band').values
    n_rows, n_cols, _ = Rrs_arr_wq.shape
    n_water = int(np.isfinite(Rrs_arr_wq).all(axis=-1).sum())
    print(f'  Scene: {n_rows}×{n_cols}  ({n_water} water pixels)')

    t0 = time.perf_counter()
    results_wq = superpixel_engine.invert_image_superpixel(
        Rrs_arr_wq, setup_wq, NOISE_WQ,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SLIC_SIGMA,
        k=K,
        n_components=N_COMPONENTS,
        invert_fn=dask_engine.invert_image,
        store_sp_results=False,
        n_iter=N_ITER,
        tile_size=TILE_SIZE,
    )
    t_total = time.perf_counter() - t0
    print(f'  Step 2 OE done in {t_total:.1f}s  ({len(results_wq["sp_counts"])} segments)')

    fit_names_wq = results_wq['fit_names']
    coords_2d = {'y': _rrs_da.y, 'x': _rrs_da.x}
    sp_dims   = ('y', 'x', 'param')
    sp_coords = {**coords_2d, 'param': fit_names_wq}

    wq_ds = xr.Dataset({
        'x_hat':           xr.DataArray(results_wq['x_hat'],  dims=sp_dims, coords=sp_coords),
        'sigma':           xr.DataArray(results_wq['sigma'],  dims=sp_dims, coords=sp_coords),
        'A_diag':          xr.DataArray(results_wq['A_diag'], dims=sp_dims, coords=sp_coords),
        'chi2':            xr.DataArray(results_wq['chi2'],            dims=('y', 'x'), coords=coords_2d),
        'chi2_calibrated': xr.DataArray(results_wq['chi2_calibrated'], dims=('y', 'x'), coords=coords_2d),
        'H_info':          xr.DataArray(results_wq['H_info'],          dims=('y', 'x'), coords=coords_2d),
    }).rio.write_crs(scene_crs)
    store.write_data(wq_ds,
                     f'{OUTPUT_PREFIX}{scene_id}-wq_params_oe_sp.zarr', replace=True)
    print(f'  Saved.')

## Step 2 diagnostics — noise calibration (OE)

Chi2 target: **≈ 1** (same as per-pixel OE).

In [ ]:
_scene = SCENE_IDS[0]
_wq   = store.open_data(f'{OUTPUT_PREFIX}{_scene}-wq_params_oe_sp.zarr')
_chi2_cal = _wq['chi2_calibrated'].values
_med      = float(np.nanmedian(_chi2_cal[np.isfinite(_chi2_cal)]))

noise_cal = NOISE_WQ * np.sqrt(_med)   # OE target = 1

print(f'NOISE_WQ = {NOISE_WQ:.5f} sr⁻¹')
print(f'Median chi2_calibrated = {_med:.3f}  →  calibrated NOISE_WQ = {noise_cal:.5f} sr⁻¹')
if abs(_med - 1.0) > 0.1:
    print(f'→ Update NOISE_WQ = {noise_cal:.5f} and re-run.')
else:
    print('✓ chi2_calibrated ≈ 1 — noise well-calibrated.')

## WQ maps — OE superpixel

In [ ]:
_scene = SCENE_IDS[0]
_wq = store.open_data(f'{OUTPUT_PREFIX}{_scene}-wq_params_oe_sp.zarr')
_valid    = np.isfinite(_wq['x_hat']).any('param')
_x_hat    = _wq['x_hat'].where(_valid)
_sigma    = _wq['sigma'].where(_valid)
_A_diag   = _wq['A_diag'].where(_valid)
_chi2_cal = _wq['chi2_calibrated'].where(_valid)

_opts   = dict(x='x', y='y', aspect='equal', width=280, height=280)
_opts_r = dict(**_opts, robust=True)

p_chi2    = _chi2_cal.hvplot(cmap='RdYlGn_r', clim=(0, 3), title='chi2_calibrated (target ≈ 1)', **_opts)
p_dfs     = _A_diag.mean('param').hvplot(cmap='viridis', clim=(0, 1), title='mean A_diag (DFS/n)', **_opts)
p_chl     = _x_hat.sel(param='C_0').hvplot(  cmap='YlGn',    clim=(0, 50),  title='C_0 Chl-a [mg/m³]', **_opts)
p_cdom    = _x_hat.sel(param='C_Y').hvplot(  cmap='YlOrBr',  clim=(0, 2),   title='C_Y CDOM [m⁻¹]',   **_opts)
p_spm     = _x_hat.sel(param='C_X').hvplot(  cmap='hot_r',   clim=(0, 10),  title='C_X SPM [g/m³]',    **_opts)
p_zB      = _x_hat.sel(param='zB').hvplot(   cmap='Blues_r', clim=(0, 20),  title='zB depth [m]',       **_opts)
p_sig_chl = _sigma.sel(param='C_0').hvplot(cmap='Purples', title='σ(C_0 Chl-a)', **_opts_r)
p_sig_zB  = _sigma.sel(param='zB').hvplot( cmap='Purples', title='σ(zB depth)',  **_opts_r)

hv.Layout([p_chi2, p_dfs, p_chl, p_cdom, p_spm, p_zB, p_sig_chl, p_sig_zB]).cols(4)